# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
# Print fields such as version and publication date
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All references to record sets, fields, and columns use their `@id` as required by the Croissant schema. This ensures consistency and clarity when handling entities in this dataset.

In [ ]:
# List all record sets and their fields/columns by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', '(no name)')}")
    if 'field' in rs:
        print(f"  Fields @id:")
        for field in rs['field']:
            print(f"    - {field['@id']} (name: {field.get('name', '(no name)')})")
    elif 'column' in rs:
        print(f"  Columns @id:")
        for col in rs['column']:
            print(f"    - {col['@id']} (name: {col.get('name', '(no name)')})")
    else:
        print("  (No fields or columns defined)")
print("\nSample records from the main record set:")
# Choose the first record set and print a few records by @id
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    for idx, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(rec)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset there is typically a single core RecordSet representing the survey table (looked up by `@id`).

In [ ]:
# Extract data from each record set
import collections.abc

dataframes = {}
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Inspect to find main record set @id
print(f"Record set @ids: {record_set_ids}")
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} rows for RecordSet {rs_id}")

# Choose main recordset to explore (the first for demonstration)
main_rs_id = record_set_ids[0]
print(f"Available columns (@id) for main record set ({main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on a numeric field of interest, such as a mental health assessment score (e.g., GAD-7 or PHQ-9).
- Normalize numeric fields.
- Group by a demographic column (e.g., gender or village).

> All field or column names below are referenced by their `@id` as per the Croissant standard. Update the variables below to match the IDs from the printed record set (section 2/3).

In [ ]:
# Replace these with the true @ids from your record set
# Example @id values (update to your schema as needed)
numeric_field_id = 'cr:PHQ9_score'   # Update with actual PHQ-9 score @id
group_field_id = 'cr:gender'        # Update with actual gender field @id

df = dataframes[main_rs_id]
if numeric_field_id in df.columns:
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception as e:
        print(f"Could not convert {numeric_field_id} to numeric: {e}")

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print(f"Group field {group_field_id} not found in columns: {df.columns.tolist()}")
else:
    print(f"Numeric field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following example shows a histogram of the PHQ-9 score (`cr:PHQ9_score`), and a boxplot of score grouped by gender (`cr:gender`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
if numeric_field_id in df.columns:
    # Histogram
    df[numeric_field_id].dropna().astype(float).hist(bins=16)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by demographic
    if group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated programmatic access to the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We explored schema metadata, loaded record sets by their `@id`, performed filtering, normalization, grouping, and displayed visualizations for key fields (e.g., PHQ-9 scores by gender). This approach supports flexible FAIR data exploration and can be extended to analytical workflows and machine learning experiments.

> **Note**: Replace example field/column `@id` values (e.g., `'cr:PHQ9_score'`, `'cr:gender'`) as needed to match the schema. Use the printout from sections 2 & 3 to find the correct `@id`s for your use case.